<style>
.jp-RenderedHTMLCommon h1 { color:#ff9900; font-size:2.3em; }
.jp-RenderedHTMLCommon h2 { color:#2563a8; }
.jp-RenderedHTMLCommon blockquote { border-left:6px solid #ff9900; background:#fff7e8; padding:.65em 1em; }
.jp-RenderedHTMLCommon code { color:#9a3412; }
</style>

# MovieLens Athena practical
## Silver Glue tables → popular-ratings view → gold table

This practical queries the Parquet silver tables created by `D345_MovieLens_Movies_Silver.ipynb` and `D346_MovieLens_Ratings_Silver.ipynb`, then materializes highly rated, sufficiently popular movies as a gold Parquet table.

# Assumptions and naming

- Athena engine version 3 and the `AwsDataCatalog` catalog are used.
- Glue database: `movielens`.
- Physical silver tables from the Glue notebooks: `movies_silver` and `ratings_silver`.
- Requested query names: `movielens_movies` and `movielens_ratings`. This notebook creates them as Athena compatibility views over the physical silver tables.
- Gold destination: `s3://gksdatalake/gold/movielens/popular_movies/`. It must be empty before the CTAS statement runs.
- The Athena workgroup must have a query-results S3 location and permission to read silver data, write the gold prefix, and update the Glue Catalog.

> Athena has persistent views, not Spark-style session temporary views. `CREATE OR REPLACE VIEW` is used so the aggregation can be queried again and consumed by CTAS. Run each SQL block separately in the Athena query editor.

# 1 — Confirm the source tables and schemas

```sql
SHOW TABLES IN movielens;
```

```sql
SHOW COLUMNS FROM movielens.movies_silver;
```

```sql
SHOW COLUMNS FROM movielens.ratings_silver;
```

Expected essential columns:

- `movies_silver`: `movie_id`, `title`, `release_year`, `genres`, `genres_raw`, `ingested_at_utc`
- `ratings_silver`: `user_id`, `movie_id`, `rating`, `rating_epoch`, `rated_at_utc`, `event_date`, `ingested_at_utc`, `event_year`

# 2 — Create the requested MovieLens names over silver

These views keep the requested table-like names while guaranteeing that the source is the silver output rather than the original CSV/bronze tables.

```sql
CREATE OR REPLACE VIEW movielens.movielens_movies AS
SELECT
    movie_id,
    title,
    release_year,
    genres,
    genres_raw
FROM movielens.movies_silver;
```

```sql
CREATE OR REPLACE VIEW movielens.movielens_ratings AS
SELECT
    user_id,
    movie_id,
    rating,
    rating_epoch,
    rated_at_utc,
    event_date,
    event_year
FROM movielens.ratings_silver;
```

# 3 — List movies and ratings

Always use a `LIMIT` for initial inspection. It limits returned rows, though it does not necessarily limit bytes scanned.

```sql
SELECT movie_id, title, release_year, genres_raw
FROM movielens.movielens_movies
ORDER BY movie_id
LIMIT 20;
```

```sql
SELECT user_id, movie_id, rating, rated_at_utc, event_date
FROM movielens.movielens_ratings
ORDER BY rated_at_utc DESC
LIMIT 20;
```

# 4 — Create the popular-ratings view

A movie qualifies when its average rating is at least `4.0` and it has at least `100` valid silver ratings. Aggregates are repeated in `HAVING` rather than relying on SELECT aliases, which keeps the statement portable and unambiguous.

```sql
CREATE OR REPLACE VIEW movielens.popular_rated_movies AS
SELECT
    movie_id,
    AVG(rating) AS avg_rating,
    COUNT(*) AS total_ratings
FROM movielens.movielens_ratings
GROUP BY movie_id
HAVING AVG(rating) >= 4.0
   AND COUNT(*) >= 100;
```

Preview the aggregated view:

```sql
SELECT movie_id, avg_rating, total_ratings
FROM movielens.popular_rated_movies
ORDER BY total_ratings DESC, avg_rating DESC, movie_id
LIMIT 50;
```

# 5 — Validate the join before writing

This detects a broken key relationship before CTAS creates files. `unmatched_movies` should be zero.

```sql
SELECT
    COUNT(*) AS popular_rows,
    COUNT(m.movie_id) AS matched_movies,
    COUNT_IF(m.movie_id IS NULL) AS unmatched_movies
FROM movielens.popular_rated_movies p
LEFT JOIN movielens.movielens_movies m
    ON m.movie_id = p.movie_id;
```

Inspect the exact gold-producing SELECT:

```sql
SELECT
    m.movie_id,
    m.title,
    m.release_year,
    m.genres,
    m.genres_raw,
    p.avg_rating,
    p.total_ratings
FROM movielens.popular_rated_movies p
INNER JOIN movielens.movielens_movies m
    ON m.movie_id = p.movie_id
ORDER BY p.total_ratings DESC, p.avg_rating DESC, m.movie_id;
```

# 6 — Explain the gold-producing query

`EXPLAIN` validates planning without executing the full data-producing CTAS.

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT
    m.movie_id,
    m.title,
    m.release_year,
    m.genres,
    m.genres_raw,
    p.avg_rating,
    p.total_ratings
FROM movielens.popular_rated_movies p
INNER JOIN movielens.movielens_movies m
    ON m.movie_id = p.movie_id;
```

# 7 — Create `gold_popular_movies` with CTAS

Run this once against an empty S3 prefix. The lowercase property name `format` is required by Athena CTAS. Parquet plus Snappy keeps the gold table columnar and compressed.

```sql
CREATE TABLE movielens.gold_popular_movies
WITH (
    format = 'PARQUET',
    write_compression = 'SNAPPY',
    external_location = 's3://gksdatalake/gold/movielens/popular_movies/'
) AS
SELECT
    m.movie_id,
    m.title,
    m.release_year,
    m.genres,
    m.genres_raw,
    p.avg_rating,
    p.total_ratings
FROM movielens.popular_rated_movies p
INNER JOIN movielens.movielens_movies m
    ON m.movie_id = p.movie_id;
```

> If the workgroup enforces its own query-results location, Athena rejects CTAS with `external_location`. In that case, remove only the `external_location` property and let the workgroup generate a unique table path.

# 8 — Validate the gold table

```sql
SHOW CREATE TABLE movielens.gold_popular_movies;
```

```sql
SELECT
    movie_id,
    title,
    release_year,
    genres_raw,
    ROUND(avg_rating, 3) AS avg_rating,
    total_ratings
FROM movielens.gold_popular_movies
ORDER BY total_ratings DESC, avg_rating DESC, movie_id;
```

```sql
SELECT
    COUNT(*) AS gold_rows,
    MIN(avg_rating) AS minimum_average,
    MIN(total_ratings) AS minimum_rating_count,
    COUNT(DISTINCT movie_id) AS distinct_movies
FROM movielens.gold_popular_movies;
```

Expected invariants: `minimum_average >= 4.0`, `minimum_rating_count >= 100`, and `gold_rows = distinct_movies`.

# Safe rerun procedure

A Hive CTAS table is not overwritten in place. `DROP TABLE` removes Catalog metadata but does **not** delete its S3 objects. To rebuild:

1. Run `DROP TABLE IF EXISTS movielens.gold_popular_movies;`.
2. Delete only the objects under the exact dedicated prefix `s3://gksdatalake/gold/movielens/popular_movies/`.
3. Confirm that prefix is empty.
4. Rerun the CTAS statement.

Do not point CTAS at either silver prefix, the Athena query-results prefix, or a shared parent folder.

# Local dataset note

The requested `C:\dataset` directory is not present in the current environment. The inspected MovieLens copy is located at:

```text
C:\data\movielens\ml-latest-small\movies\movies.csv
C:\data\movielens\ml-latest-small\ratings\ratings.csv
C:\data\movielens\parquet\movies.parquet
C:\data\movielens\parquet\ratings.parquet
```

The raw headers are `movieId,title,genres` and `userId,movieId,rating,timestamp`. The Glue silver notebooks deliberately normalize these to `movie_id`, `user_id`, `rating`, and typed timestamp/date columns used above.

# Verification notes and official references

The SQL follows Athena's documented `CREATE OR REPLACE VIEW`, `SHOW COLUMNS`, `EXPLAIN`, and Hive CTAS syntax. Important verified constraints are that CTAS `format` is lowercase, an explicit `external_location` must be empty, Athena does not delete an existing destination, and an enforced workgroup output setting is incompatible with an explicit CTAS `external_location`.

- [CREATE VIEW](https://docs.aws.amazon.com/athena/latest/ug/create-view.html)
- [CREATE TABLE AS](https://docs.aws.amazon.com/athena/latest/ug/create-table-as.html)
- [CTAS format and compression examples](https://docs.aws.amazon.com/athena/latest/ug/ctas-examples.html)
- [SHOW COLUMNS](https://docs.aws.amazon.com/athena/latest/ug/show-columns.html)
- [Workgroup CTAS troubleshooting](https://docs.aws.amazon.com/athena/latest/ug/workgroups-troubleshooting.html)

> Final execution still depends on the actual AWS Region, Catalog contents, Lake Formation/IAM grants, S3 bucket policy, KMS settings, and workgroup configuration.